# BioDPR Index Creation

By successfully executing the code in the following cells, you will create an index for bioDPR for the experiment based on the TREC-COVID dataset.

In [5]:
import pyterrier as pt
import os
from pathlib import Path
import warnings
import pyterrier_dr as ptdr

warnings.filterwarnings('ignore', category=DeprecationWarning)

In [6]:
dataset_name = "irds:cord19/trec-covid"
dataset = pt.get_dataset(dataset_name)

In [7]:
biodpr_model_name = "pritamdeka/S-PubMedBert-MS-MARCO"
biodpr_model = ptdr.SBertBiEncoder(biodpr_model_name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: pritamdeka/S-PubMedBert-MS-MARCO
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
index_dir_path = Path("../../index/trec_covid_biodpr_complete").resolve()
index_dir = str(index_dir_path)
index_properties_file = os.path.join(index_dir, "pt_meta.json")
os.makedirs(index_dir, exist_ok=True)

In [18]:
# This generator function will be used to create the sparse index by concatenating the title and abstract of each document in the TREC-COVID dataset.
def trec_covid_corpus_generator():
    for doc in dataset.get_corpus_iter():
        title = str(doc.get('title', ''))
        abstract = str(doc.get('abstract', ''))

        yield {
            'docno': doc['docno'],
            'text': title + " " + abstract
        }

In [19]:
if os.path.exists(index_properties_file):
    print(f"Index found at {index_dir}. You are ready to go!")
else:
    print(f"Index not found at {index_dir}. Building it now...")
    flex_index = ptdr.FlexIndex(index_dir)
    indexing_pipeline = biodpr_model >> flex_index.indexer(mode="overwrite") 
    indexing_pipeline.index(trec_covid_corpus_generator())

Index found at C:\Users\zosia\Desktop\Studies\Information Retrieval\IR_Project\index\trec_covid_biodpr_complete. You are ready to go!
